# ViT-aligned ResNet-50 CNN baseline using dataset-team CSV splits

This notebook uses the exact dataset-team `train.csv`, `val.csv`, and `test.csv` rows. For the no-augmentation baseline, its spatial preprocessing is aligned with the CLIP ViT pipeline: bicubic resize to 224 followed by a 224 center crop. ResNet retains ImageNet normalization because its pretrained weights expect different statistics than CLIP.

- `experiments.json` supplies experiment metadata.
- `_experiment_manifests/<experiment>/train.csv` updates model weights.
- `_experiment_manifests/<experiment>/val.csv` selects the best checkpoint.
- `_experiment_manifests/<experiment>/test.csv` supplies final metrics.
- Google Drive paths are remapped to the cluster's local `GenImage_Compact` root.


In [ ]:
# Run only if packages are missing.
# %pip install -q torch torchvision pandas numpy scikit-learn matplotlib seaborn pillow tqdm

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import random
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image, ImageFile
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.models import ResNet50_Weights, resnet50
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Configuration

In [ ]:
@dataclass
class Config:
    data_root: str = "/project2/yzhao010_1531/csci566-genimage/data/extracted/GenImage_Compact"
    experiment_root: str = "/project2/yzhao010_1531/csci566-genimage/data/extracted/GenImage_Compact/benchmark/manifests"
    output_root: str = "./cnn_outputs"

    image_size: int = 224
    batch_size: int = 64
    num_workers: int = 6
    epochs: int = 15
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    early_stopping_patience: int = 4
    pretrained: bool = True
    use_amp: bool = True
    decision_threshold: float = 0.5
    base_seed: int = 17

    # Optional fast integration test. It temporarily subsamples the supplied CSVs.
    # Keep None for real experiments so the exact dataset-team splits are used.
    debug_max_rows_per_split: Optional[int] = None


CFG = Config()

# One or more metadata files; each must contain a JSON list of experiments.
EXPERIMENT_PLAN_FILES: list[str] = [
    f"{CFG.experiment_root}/experiments.json",
]

# ["ALL"] runs everything. Otherwise list any one or several exact names.
SELECTED_EXPERIMENTS: list[str] = [
    "holdout_ADM_k1_rep0",
    # "holdout_ADM_k2_rep0",
]

# Choose either variant separately or both. Order controls execution order.
TRAINING_VARIANTS: list[str] = [
    "baseline",
    # "augmented",
]

print(asdict(CFG))
print("Plan files:", EXPERIMENT_PLAN_FILES)
print("Requested experiments:", SELECTED_EXPERIMENTS)

The `counts` values in `experiments.json` are checked against the number of rows in the supplied CSVs. No interpretation or 50/50 resampling is performed.

## 2. Reproducibility and CSV path remapping

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def relative_genimage_path(raw_path: str) -> str:
    text = str(raw_path).replace("\\", "/")
    marker = re.search(r"(?:^|/)GenImage_Compact(?:_shared)?/", text)
    if marker:
        return text[marker.end():]

    path = Path(text)
    if not path.is_absolute():
        return path.as_posix()

    try:
        return path.relative_to(Path(CFG.data_root)).as_posix()
    except ValueError as exc:
        raise ValueError(f"Cannot map CSV path to GenImage_Compact: {raw_path}") from exc


def debug_cap(frame: pd.DataFrame, maximum: Optional[int], seed: int) -> pd.DataFrame:
    if maximum is None or len(frame) <= maximum:
        return frame.copy()
    # Debug-only stratification; full runs always preserve the CSV exactly.
    parts = []
    labels = sorted(frame.label.unique())
    base, remainder = divmod(maximum, len(labels))
    for i, label in enumerate(labels):
        count = base + int(i < remainder)
        pool = frame[frame.label == label]
        parts.append(pool.sample(n=min(count, len(pool)), random_state=seed + i))
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed)


def read_split_csv(csv_path: Path, split_name: str, seed: int) -> pd.DataFrame:
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing experiment split: {csv_path}")
    frame = pd.read_csv(csv_path)
    required = {"path", "label", "source_generator", "source_split", "sha256"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing columns: {sorted(missing)}")

    frame = frame.copy()
    frame["original_path"] = frame["path"].astype(str)
    frame["path"] = frame["original_path"].map(relative_genimage_path)
    frame["label"] = pd.to_numeric(frame["label"], errors="raise").astype(int)
    if not set(frame.label.unique()).issubset({0, 1}):
        raise ValueError(f"{csv_path}: labels must be 0 or 1.")
    frame["label_name"] = frame.label.map({0: "nature", 1: "ai"})
    frame["generator"] = frame["source_generator"].astype(str)
    frame["official_split"] = frame["source_split"].astype(str).str.lower()
    frame["experiment_split"] = split_name

    if frame.sha256.isna().any() or frame.sha256.duplicated().any():
        raise ValueError(f"{csv_path}: missing or duplicate sha256 values within this split.")

    missing_files = [
        str(Path(CFG.data_root) / rel)
        for rel in frame.path
        if not (Path(CFG.data_root) / rel).is_file()
    ]
    if missing_files:
        preview = missing_files[:5]
        raise FileNotFoundError(
            f"{csv_path}: {len(missing_files)} remapped image files do not exist. "
            f"First examples: {preview}"
        )

    return debug_cap(frame, CFG.debug_max_rows_per_split, seed)

## 3. Load and select experiment specifications

In [ ]:
def load_experiment_plans(plan_files: Sequence[str]) -> list[dict]:
    if isinstance(plan_files, str):
        raise TypeError("EXPERIMENT_PLAN_FILES must be a list of strings, not one string.")
    experiments = []
    for raw_path in plan_files:
        if not isinstance(raw_path, str):
            raise TypeError("Every plan-file entry must be a string.")
        path = Path(raw_path)
        with path.open() as file:
            entries = json.load(file)
        if not isinstance(entries, list):
            raise ValueError(f"{path} must contain a JSON list.")
        for entry in entries:
            if not isinstance(entry, dict):
                raise ValueError(f"Every entry in {path} must be a JSON object.")
            entry = dict(entry)
            entry["_plan_file"] = str(path)
            experiments.append(entry)

    names = [entry.get("experiment") for entry in experiments]
    duplicates = pd.Series(names).duplicated(keep=False)
    if duplicates.any():
        repeated = sorted(set(pd.Series(names)[duplicates]))
        raise ValueError(f"Duplicate experiment names across plans: {repeated}")
    return experiments


def validate_experiment(entry: dict, available: Sequence[str]) -> None:
    required = {
        "experiment", "holdout", "k", "rep", "train_generators",
        "test_generator", "nature_generator", "counts",
    }
    missing = required - set(entry)
    if missing:
        raise ValueError(f"{entry.get('experiment', '<unnamed>')} is missing {sorted(missing)}")
    if not isinstance(entry["train_generators"], list) or not all(
        isinstance(x, str) for x in entry["train_generators"]
    ):
        raise TypeError("train_generators must be a list of strings.")
    if len(entry["train_generators"]) != int(entry["k"]):
        raise ValueError(f"{entry['experiment']}: k does not match train_generators length.")
    if len(set(entry["train_generators"])) != len(entry["train_generators"]):
        raise ValueError(f"{entry['experiment']}: duplicate training generators.")
    if entry["holdout"] in entry["train_generators"]:
        raise ValueError(f"{entry['experiment']}: holdout appears in training generators.")
    if entry["test_generator"] in entry["train_generators"]:
        raise ValueError(f"{entry['experiment']}: test generator appears in training generators.")
    referenced = set(entry["train_generators"]) | {
        entry["test_generator"], entry["nature_generator"]
    }
    unknown = referenced - set(available)
    if unknown:
        raise ValueError(f"{entry['experiment']}: unknown generators {sorted(unknown)}")
    if set(entry["counts"]) != {"train", "val", "test"}:
        raise ValueError(f"{entry['experiment']}: counts must contain train, val, and test.")
    if any(int(value) <= 0 for value in entry["counts"].values()):
        raise ValueError(f"{entry['experiment']}: every count must be positive.")


available_generators = sorted(
    p.name for p in Path(CFG.data_root).iterdir()
    if p.is_dir() and not p.name.startswith("_")
)
all_experiments = load_experiment_plans(EXPERIMENT_PLAN_FILES)
for experiment in all_experiments:
    validate_experiment(experiment, available_generators)

if not isinstance(SELECTED_EXPERIMENTS, list) or not all(
    isinstance(name, str) for name in SELECTED_EXPERIMENTS
):
    raise TypeError("SELECTED_EXPERIMENTS must be a list of strings.")

if len(SELECTED_EXPERIMENTS) == 1 and SELECTED_EXPERIMENTS[0].upper() == "ALL":
    selected_experiments = all_experiments
else:
    requested = set(SELECTED_EXPERIMENTS)
    known = {entry["experiment"] for entry in all_experiments}
    unknown = requested - known
    if unknown:
        raise ValueError(f"Unknown requested experiment names: {sorted(unknown)}")
    selected_experiments = [
        entry for entry in all_experiments if entry["experiment"] in requested
    ]

if not selected_experiments:
    raise ValueError("No experiments selected.")

print(f"Selected {len(selected_experiments)} of {len(all_experiments)} experiments:")
for entry in selected_experiments:
    print(" -", entry["experiment"])

## 4. Load and validate the exact experiment CSV splits

In [ ]:
def load_experiment_splits(entry: dict):
    seed = CFG.base_seed
    experiment_dir = Path(CFG.experiment_root) / entry["experiment"]
    train_df = read_split_csv(experiment_dir / "train.csv", "train", seed + 10)
    val_df = read_split_csv(experiment_dir / "val.csv", "val", seed + 20)
    test_df = read_split_csv(experiment_dir / "test.csv", "test", seed + 30)

    # Exact row counts are enforced for full runs. Debug mode intentionally caps them.
    if CFG.debug_max_rows_per_split is None:
        actual = {"train": len(train_df), "val": len(val_df), "test": len(test_df)}
        expected = {name: int(entry["counts"][name]) for name in ("train", "val", "test")}
        if actual != expected:
            raise ValueError(
                f"{entry['experiment']}: CSV row counts {actual} do not match plan counts {expected}."
            )

    # Hashes—not path strings—guard against leakage and duplicate content.
    for split_name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
        if frame.sha256.duplicated().any():
            raise ValueError(f"{entry['experiment']}: duplicate content inside {split_name}.csv")
    if set(train_df.sha256) & set(val_df.sha256):
        raise ValueError(f"{entry['experiment']}: train/validation content leakage detected.")
    if set(train_df.sha256) & set(test_df.sha256):
        raise ValueError(f"{entry['experiment']}: train/test content leakage detected.")
    if set(val_df.sha256) & set(test_df.sha256):
        raise ValueError(f"{entry['experiment']}: validation/test content leakage detected.")

    train_ai_generators = set(train_df.loc[train_df.label == 1, "generator"])
    val_ai_generators = set(val_df.loc[val_df.label == 1, "generator"])
    expected_train_generators = set(entry["train_generators"])
    if train_ai_generators != expected_train_generators:
        raise ValueError(
            f"{entry['experiment']}: train AI generators {sorted(train_ai_generators)} "
            f"do not match plan {sorted(expected_train_generators)}."
        )
    if val_ai_generators != expected_train_generators:
        raise ValueError(
            f"{entry['experiment']}: validation AI generators {sorted(val_ai_generators)} "
            f"do not match plan {sorted(expected_train_generators)}."
        )
    test_ai_generators = set(test_df.loc[test_df.label == 1, "generator"])
    if test_ai_generators != {entry["test_generator"]}:
        raise ValueError(
            f"{entry['experiment']}: test AI generators {sorted(test_ai_generators)} "
            f"do not match {entry['test_generator']}."
        )

    for split_name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
        nature_generators = set(frame.loc[frame.label == 0, "generator"])
        if nature_generators != {entry["nature_generator"]}:
            raise ValueError(
                f"{entry['experiment']}: {split_name} nature generators "
                f"{sorted(nature_generators)} do not match {entry['nature_generator']}."
            )

    # Preserve and report source_split, but keep the dataset team's CSV membership
    # authoritative. Rows are never silently moved between experiment splits.
    return train_df, val_df, test_df, seed

## 5. Images, transformations, and loaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

augmented_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        CFG.image_size, scale=(0.8, 1.0),
        interpolation=InterpolationMode.BICUBIC,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(
        CFG.image_size, interpolation=InterpolationMode.BICUBIC,
    ),
    transforms.CenterCrop(CFG.image_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

TRAIN_TRANSFORMS = {
    # Same deterministic spatial preprocessing as the current ViT baseline.
    "baseline": eval_transform,
    # Random crop and horizontal flip for the augmentation ablation.
    "augmented": augmented_train_transform,
}

for variant in TRAINING_VARIANTS:
    description = "none" if variant == "baseline" else "random resized crop + horizontal flip"
    print(f"{variant} training augmentation: {description}")


class ManifestImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, data_root: str, transform):
        self.frame = frame.reset_index(drop=True)
        self.data_root = Path(data_root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        raw_path = Path(str(row.path))
        path = raw_path if raw_path.is_absolute() else self.data_root / raw_path
        try:
            with Image.open(path) as image:
                image = image.convert("RGB")
        except Exception as exc:
            raise RuntimeError(f"Unable to load {path}") from exc
        return {
            "image": self.transform(image),
            "label": torch.tensor(float(row.label), dtype=torch.float32),
            "path": str(row.path),
            "generator": str(row.generator),
        }


def make_loader(frame, transform, seed, shuffle=False):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        ManifestImageDataset(frame, CFG.data_root, transform),
        batch_size=CFG.batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=CFG.num_workers > 0,
        worker_init_fn=worker_init_fn,
        generator=generator,
    )

## 6. Model, metrics, training, and prediction functions

In [ ]:
def build_model() -> nn.Module:
    weights = ResNet50_Weights.IMAGENET1K_V2 if CFG.pretrained else None
    model = resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model


def safe_auroc(labels, probabilities) -> float:
    return float(roc_auc_score(labels, probabilities)) if len(np.unique(labels)) == 2 else float("nan")


def binary_metrics(labels, probabilities, threshold=0.5) -> dict:
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        "n": int(len(labels)),
        "accuracy": float(accuracy_score(labels, predictions)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "auroc": safe_auroc(labels, probabilities),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss, labels, probabilities = 0.0, [], []
    for batch in tqdm(loader, leave=False):
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["label"].to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training), autocast(enabled=CFG.use_amp and DEVICE.type == "cuda"):
            logits = model(images).squeeze(1)
            loss = criterion(logits, targets)
        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * len(targets)
        labels.extend(targets.detach().cpu().numpy())
        probabilities.extend(torch.sigmoid(logits).detach().cpu().numpy())
    metrics = binary_metrics(labels, probabilities, CFG.decision_threshold)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


def predict(model, loader) -> pd.DataFrame:
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            images = batch["image"].to(DEVICE, non_blocking=True)
            with autocast(enabled=CFG.use_amp and DEVICE.type == "cuda"):
                probs = torch.sigmoid(model(images).squeeze(1)).cpu().numpy()
            for path, generator, label, prob in zip(
                batch["path"], batch["generator"], batch["label"].numpy(), probs
            ):
                rows.append({
                    "path": path, "generator": generator, "label": int(label),
                    "probability_fake": float(prob),
                })
    result = pd.DataFrame(rows)
    result["prediction"] = (result.probability_fake >= CFG.decision_threshold).astype(int)
    return result

## 7. Execute one experiment

In [ ]:
def run_experiment(entry: dict, variant: str) -> dict:
    if variant not in TRAIN_TRANSFORMS:
        raise ValueError(f"Unknown training variant: {variant}")
    base_experiment = entry["experiment"]
    run_name = f"{base_experiment}__{variant}"
    train_transform = TRAIN_TRANSFORMS[variant]
    run_dir = Path(CFG.output_root) / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    train_df, val_df, test_df, seed = load_experiment_splits(entry)
    seed_everything(seed)

    print("\n" + "=" * 80)
    print(run_name)
    print("Variant:", variant)
    print("Training augmentation:", "none" if variant == "baseline" else "random crop + flip")
    print("Train generators:", entry["train_generators"])
    print("Test generator:", entry["test_generator"])
    print("Seed:", seed)
    print("Exact CSV split sizes:", {"train": len(train_df), "val": len(val_df), "test": len(test_df)})
    print("Train table:\n", pd.crosstab(train_df.generator, train_df.label_name, margins=True))
    print("Validation table:\n", pd.crosstab(val_df.generator, val_df.label_name, margins=True))
    print("Test table:\n", pd.crosstab(test_df.generator, test_df.label_name, margins=True))
    split_audit = pd.concat([train_df, val_df, test_df]).groupby(["experiment_split", "official_split"]).size()
    print("CSV/source-split audit:\n", split_audit)

    train_df.to_csv(run_dir / "train_split.csv", index=False)
    val_df.to_csv(run_dir / "validation_split.csv", index=False)
    test_df.to_csv(run_dir / "test_split.csv", index=False)
    metadata = {
        **entry, "run_name": run_name, "variant": variant,
        "augmentation": "none" if variant == "baseline" else "random_resized_crop+horizontal_flip",
        "resolved_seed": seed, "config": asdict(CFG),
    }
    (run_dir / "experiment.json").write_text(json.dumps(metadata, indent=2))

    train_loader = make_loader(train_df, train_transform, seed, shuffle=True)
    val_loader = make_loader(val_df, eval_transform, seed, shuffle=False)
    test_loader = make_loader(test_df, eval_transform, seed, shuffle=False)

    model = build_model().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
    scaler = GradScaler(enabled=CFG.use_amp and DEVICE.type == "cuda")

    history, best_auroc, no_improvement = [], -np.inf, 0
    for epoch in range(1, CFG.epochs + 1):
        start = time.time()
        train_metrics = run_epoch(model, train_loader, criterion, optimizer, scaler)
        val_metrics = run_epoch(model, val_loader, criterion)
        row = {
            "epoch": epoch, "learning_rate": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - start,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        checkpoint = {
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_auroc": best_auroc, "experiment": entry,
            "run_name": run_name, "variant": variant,
            "augmentation": "none" if variant == "baseline" else "random_resized_crop+horizontal_flip",
            "config": asdict(CFG), "resolved_seed": seed,
        }
        torch.save(checkpoint, run_dir / "last.pt")

        score = val_metrics["auroc"]
        if np.isfinite(score) and score > best_auroc:
            best_auroc, no_improvement = score, 0
            checkpoint["best_val_auroc"] = best_auroc
            torch.save(checkpoint, run_dir / "best.pt")
        else:
            no_improvement += 1
        scheduler.step()
        print(
            f"Epoch {epoch:02d}: train loss={train_metrics['loss']:.4f}, "
            f"val loss={val_metrics['loss']:.4f}, val F1={val_metrics['f1']:.4f}, "
            f"val AUROC={val_metrics['auroc']:.4f}"
        )
        if no_improvement >= CFG.early_stopping_patience:
            print("Early stopping")
            break

    best = torch.load(run_dir / "best.pt", map_location=DEVICE)
    model.load_state_dict(best["model_state_dict"])
    predictions = predict(model, test_loader)
    predictions.to_csv(run_dir / "test_predictions.csv", index=False)
    test_metrics = binary_metrics(
        predictions.label, predictions.probability_fake, CFG.decision_threshold
    )
    result = {
        "run_name": run_name,
        "experiment": base_experiment,
        "variant": variant,
        "augmentation": "none" if variant == "baseline" else "random_resized_crop+horizontal_flip",
        "holdout": entry["holdout"],
        "test_generator": entry["test_generator"],
        "nature_generator": entry["nature_generator"],
        "k": int(entry["k"]),
        "rep": int(entry["rep"]),
        "seed": seed,
        "train_generators": "+".join(entry["train_generators"]),
        "best_epoch": int(best["epoch"]),
        "best_val_auroc": float(best["best_val_auroc"]),
        **{f"test_{key}": value for key, value in test_metrics.items()},
    }
    (run_dir / "summary.json").write_text(json.dumps(result, indent=2))

    history_df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history_df.epoch, history_df.train_loss, label="train")
    axes[0].plot(history_df.epoch, history_df.val_loss, label="validation")
    axes[0].set(title="Loss", xlabel="Epoch")
    axes[1].plot(history_df.epoch, history_df.train_auroc, label="train")
    axes[1].plot(history_df.epoch, history_df.val_auroc, label="validation")
    axes[1].set(title="AUROC", xlabel="Epoch", ylim=(0, 1))
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    fig.suptitle(run_name)
    fig.tight_layout()
    fig.savefig(run_dir / "training_curves.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    del model, optimizer, scheduler, scaler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

## 8. Run the selected list

In [ ]:
completed_results = []
for experiment in selected_experiments:
    for variant in TRAINING_VARIANTS:
        try:
            completed_results.append(run_experiment(experiment, variant))
        except Exception as exc:
            print(f"FAILED: {experiment['experiment']} [{variant}]: {exc}")
            raise

results_df = pd.DataFrame(completed_results)
Path(CFG.output_root).mkdir(parents=True, exist_ok=True)
results_df.to_csv(Path(CFG.output_root) / "selected_experiment_results.csv", index=False)
display(results_df)

## 9. Consolidate every completed run

In [ ]:
summary_files = sorted(Path(CFG.output_root).glob("*/summary.json"))
if summary_files:
    all_results = pd.DataFrame([json.loads(path.read_text()) for path in summary_files])
    all_results.to_csv(Path(CFG.output_root) / "all_experiment_results.csv", index=False)
    display(all_results.sort_values(["holdout", "k", "variant", "rep"]))

    aggregate = (
        all_results.groupby(["holdout", "k", "variant"], as_index=False)
        .agg(
            mean_test_accuracy=("test_accuracy", "mean"),
            std_test_accuracy=("test_accuracy", "std"),
            mean_test_f1=("test_f1", "mean"),
            std_test_f1=("test_f1", "std"),
            mean_test_auroc=("test_auroc", "mean"),
            std_test_auroc=("test_auroc", "std"),
            n_runs=("run_name", "count"),
        )
    )
    aggregate.to_csv(Path(CFG.output_root) / "aggregate_by_holdout_k_and_variant.csv", index=False)
    display(aggregate)
else:
    print("No completed summaries found.")

## How to choose runs

One run:

```python
SELECTED_EXPERIMENTS = ["holdout_ADM_k1_rep0"]
```

Two runs:

```python
SELECTED_EXPERIMENTS = [
    "holdout_ADM_k1_rep0",
    "holdout_ADM_k2_rep0",
]
```

Every entry from every supplied plan file:

```python
SELECTED_EXPERIMENTS = ["ALL"]
```

For a quick integration check, first set:

```python
CFG.debug_max_rows_per_split = 200
CFG.epochs = 1
CFG.batch_size = 32
```

Then restore `debug_max_rows_per_split=None` and the full training settings.

For all reportable runs, keep:

```python
CFG.debug_max_rows_per_split = None
```

This ensures the supplied CSV rows are used exactly. Debug mode is only for a short integration check.

For all reportable runs, keep:

```python
CFG.debug_max_rows_per_split = None
```

This ensures the supplied CSV rows are used exactly. Debug mode is only for a short integration check.

## ViT/CNN alignment used here

- Exact same experiment CSV rows
- `0=real`, `1=fake`
- Bicubic resize to 224 and center crop to 224
- No random augmentation for the primary baseline
- Matched repetition seeds: 17, 18, 19
- Model-specific normalization: ImageNet for ResNet, CLIP statistics for ViT
- ResNet is fully fine-tuned from ImageNet-pretrained weights; CLIP remains frozen


## Choose baseline, augmentation, or both

Baseline only:

```python
TRAINING_VARIANTS = ["baseline"]
```

Augmented only:

```python
TRAINING_VARIANTS = ["augmented"]
```

Run both independently on every selected experiment:

```python
TRAINING_VARIANTS = ["baseline", "augmented"]
```

Outputs are isolated in directories such as:

```text
cnn_outputs/holdout_ADM_k1_rep0__baseline/
cnn_outputs/holdout_ADM_k1_rep0__augmented/
```

Both variants use the same CSV rows, seed, optimizer, learning rate, epoch limit, and evaluation transform. Only the training-image transform changes.
